# Risk/UQ Paper Track: UQ Benchmark

## Notebook Objective
This notebook evaluates uncertainty quality and shift robustness for the trained risk model and monitor/controller variants.

It converts saved training artifacts into benchmark outputs used for paper tables and figures.


## Research Questions and Hypotheses

### Primary Questions
1. Are calibrated probabilities reliable in-distribution (`nominal_clean`)?
2. How much calibration quality degrades under perturbation shifts?
3. Does calibration preserve ranking/discrimination quality (AUROC/AUPRC)?

### Hypotheses
- **H1:** calibrated monitor improves ECE over raw monitor in nominal split.
- **H2:** calibration remains beneficial on most shift suites.
- **H3:** safety-oriented selective risk curves improve at fixed coverage.


## Methodology

### Shift Suites
- `nominal_clean`
- `hist_prim_shift`
- `fut_prim_shift`
- `hist_rmv_shift`
- `high_interaction_holdout`

### Compared Variants
- raw risk probabilities
- temperature-calibrated probabilities
- optional controller tradeoff summary if base/controller scenario-level tables are provided

### Core Metrics
- calibration: ECE, adaptive ECE, NLL, Brier
- discrimination: AUROC, AUPRC
- selective prediction: coverage-risk curves
- shift analysis: gap vs nominal


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path
import pandas as pd

REPO_ROOT = Path(r'/Users/achyutmorang/Downloads/waymax research/waymax-simulation-experiments')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.closedloop import ClosedLoopConfig
from src.workflows.uq_benchmark_flow import run_uq_benchmark_flow

cfg = ClosedLoopConfig()
cfg.run_prefix = str(REPO_ROOT / 'artifacts' / 'risk_uq_pilot')
cfg.uq_eval_probability_bins = 15

cfg


In [ ]:
# Load candidate dataset created by training flow
DATASET_PATH = Path(f'{cfg.run_prefix}_risk_dataset.parquet')
if DATASET_PATH.exists():
    dataset_df = pd.read_parquet(DATASET_PATH)
else:
    dataset_df = pd.read_csv(DATASET_PATH.with_suffix('.csv'))

dataset_df.head()


In [ ]:
# Run benchmark
benchmark_bundle = run_uq_benchmark_flow(
    cfg=cfg,
    dataset_df=dataset_df,
)

benchmark_bundle.benchmark_bundle.summary_df


In [ ]:
# Optional diagnostics
# benchmark_bundle.benchmark_bundle.per_shift_df
# benchmark_bundle.benchmark_bundle.reliability_df.head()
# benchmark_bundle.benchmark_bundle.selective_curve_df.head()
# benchmark_bundle.artifact_paths


## Report Section Template: Benchmark Findings

### Calibration Outcomes
- Raw vs calibrated ECE (nominal):
- Raw vs calibrated ECE (shift average):
- Best and worst suites by calibration gap:

### Discrimination Outcomes
- AUROC / AUPRC changes after calibration:
- Any notable precision-recall degradation:

### Shift Robustness Narrative
- Which suites are most sensitive?
- Does calibrated monitor remain conservative or become overconfident?
- Implications for control-time reranking:


## Exit Criteria for This Notebook

- [ ] `*_uq_benchmark_summary.csv` written.
- [ ] `*_uq_benchmark_per_shift.csv` written.
- [ ] `*_uq_reliability_bins.csv` and `*_uq_selective_risk_curve.csv` written.
- [ ] `*_uq_shift_gap_summary.csv` written.
- [ ] Findings section filled with quantitative statements.
